Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 3.2: Table Maintenance Procedures

Traditional databases run maintenance automatically, compacting and cleaning up in the background. Iceberg separates storage (object store) from compute (Spark, Trino), so there's no always-on process to run maintenance. Instead, you schedule these procedures as periodic jobs or use a catalog service that automates them.

Each procedure addresses a different form of "waste":
- **Compact data files**: Consolidate small data files into larger ones
- **Compact manifests**: Reduce metadata overhead (fewer files to open during query planning)
- **Expire snapshots**: Clean up old table history
- **Remove orphan files**: Reclaim storage from failed operations

In this exercise, you'll learn how to maintain Apache Iceberg tables using the NYC Yellow Taxi dataset.

## Learning Objectives
- Identify when maintenance is needed
- Run data file compaction procedures
- Optimize metadata with manifest compaction
- Configure and run snapshot expiration
- Understand orphan file removal
- Monitor table health metrics

⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableMaintenance") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.maintenance")
print("Namespace 'maintenance' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June 2023**.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nTaxi data ready in MinIO!")

## Part 1: Create a Table with a Small File Problem

We'll simulate a streaming ingestion pattern by writing June taxi data in many small batches, creating a large number of small files.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.maintenance.taxi_trips")

spark.sql("""
    CREATE TABLE polaris.maintenance.taxi_trips (
        VendorID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count LONG,
        trip_distance DOUBLE,
        fare_amount DOUBLE,
        tip_amount DOUBLE,
        total_amount DOUBLE
    )
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    -- We disable manifest merging to intentionally produce fragmentation we'll fix in Part 2.
    TBLPROPERTIES (
        'write.target-file-size-bytes' = '524288',
        'commit.manifest-merge.enabled' = 'false'
    )
""")

print("Taxi trips table created!")

In [ ]:
june_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet") \
    .select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "passenger_count", "trip_distance", "fare_amount",
            "tip_amount", "total_amount")

total_rows = june_df.count()
batches = 100
rows_per_batch = total_rows // batches

print(f"Writing {total_rows:,} rows in {batches} small batches ({rows_per_batch:,} rows each)...")

for i in range(batches):
    micro = june_df.limit(rows_per_batch)
    micro.writeTo("polaris.maintenance.taxi_trips").append()
    if (i + 1) % 25 == 0:
        print(f"  Created {i + 1}/{batches} commits...")

print("\nSmall files created!")

### Assess the Table State

How do we know if a table needs maintenance? Iceberg exposes metadata tables that let you inspect the physical state of a table without reading any data:

- **`<table>.files`**: One row per data file currently in the table. Shows file path, size, record count, and column-level metrics (min/max/null counts). Use this to check file count, size distribution, and whether you have a small file problem.
- **`<table>.manifests`**: One row per manifest file. A manifest is an Avro file that indexes a group of data files along with their partition values and column statistics. Query planning must read every manifest, so too many manifests slows down planning.
- **`<table>.snapshots`**: One row per snapshot (commit). Each write, delete, or compaction creates a snapshot. Old snapshots accumulate metadata and enable time travel, but eventually need cleanup.

Let's inspect our table to see the damage from 100 small-batch commits.

In [ ]:
print("File statistics BEFORE compaction:")
spark.sql("""
    SELECT
        COUNT(*) as file_count,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
        ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
        ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.files
""").show()

In [ ]:
print("Manifest statistics:")
spark.sql("""
    SELECT
        COUNT(*) as manifest_count,
        ROUND(AVG(length) / 1024, 2) as avg_size_kb,
        ROUND(SUM(length) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.manifests
""").show()

In [ ]:
snapshot_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

**~600 small data files** (avg 0.08 MB each), **100 manifests**, and **100 snapshots**. This table needs maintenance. In the following sections we'll compact manifests to speed up planning, compact data files to speed up scans, and expire old snapshots to clean up metadata.

## Part 2: Manifest Compaction

As we saw above, our table has 100 manifests, one per commit. During query planning, Iceberg must open **every** manifest to check column statistics and decide which data files to scan. Compacting manifests merges them into fewer, larger files so the planner has less work to do.

### Check Manifest Statistics

In [ ]:
print("Manifest statistics BEFORE compaction:")
spark.sql("""
    SELECT COUNT(*) as manifest_count, ROUND(AVG(length) / 1024, 2) as avg_size_kb
    FROM polaris.maintenance.taxi_trips.manifests
""").show()

### Measure Planning Time (Before)

Query planning reads every manifest to determine which data files to scan. Even when a query matches **zero files**, the planner must still open each manifest and check column-level min/max statistics to rule files out. With 100 small manifests, that's 100 Avro files to open just for planning.

We'll use a predicate (`fare_amount > 99999`) that no data file satisfies. The planner can determine this purely from column min/max stats in the manifests, so **no data files are scanned**. This isolates manifest-reading overhead.

In [ ]:
planning_query = "SELECT COUNT(*) FROM polaris.maintenance.taxi_trips WHERE fare_amount > 99999"

plan_times_before = []
for run in range(3):
    start = time.time()
    result = spark.sql(planning_query).collect()
    plan_times_before.append(time.time() - start)

avg_before_manifest = sum(plan_times_before) / len(plan_times_before)
manifest_count = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.manifests").collect()[0][0]
print(f"Planning query (0 matching files) with {manifest_count} manifests:")
print(f"  3 runs: {', '.join(f'{t:.3f}s' for t in plan_times_before)}")
print(f"  Average: {avg_before_manifest:.3f}s")
print(f"  Rows returned: {result[0][0]} (confirmed: all files pruned via column stats)")

### Compact Manifests

In [ ]:
print("Compacting manifests...")
start = time.time()

spark.sql("""
    CALL polaris.system.rewrite_manifests(table => 'maintenance.taxi_trips')
""")

elapsed = time.time() - start
print(f"Manifest compaction completed in {elapsed:.2f} seconds")

In [ ]:
print("Manifest statistics AFTER compaction:")
spark.sql("""
    SELECT COUNT(*) as manifest_count, ROUND(AVG(length) / 1024, 2) as avg_size_kb
    FROM polaris.maintenance.taxi_trips.manifests
""").show()

In [ ]:
plan_times_after = []
for run in range(3):
    start = time.time()
    spark.sql(planning_query).collect()
    plan_times_after.append(time.time() - start)

avg_after_manifest = sum(plan_times_after) / len(plan_times_after)
manifest_count = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.manifests").collect()[0][0]
print(f"Same planning query with {manifest_count} manifest:")
print(f"  3 runs: {', '.join(f'{t:.3f}s' for t in plan_times_after)}")
print(f"  Average: {avg_after_manifest:.3f}s")
print(f"\nPlanning speedup: {avg_before_manifest/avg_after_manifest:.1f}x faster")

Fewer, larger manifest files means the query planner has fewer files to open and read during planning. The same query ran faster after manifest compaction, even though the underlying data files haven't changed yet.

Compaction created new, larger manifests, but what happened to the old ones? They're still on disk, because older snapshots still reference them.

Iceberg provides **`all_manifests`** and **`all_data_files`** metadata tables that show every file referenced by **any snapshot that hasn't been expired yet** (i.e., any snapshot you can still time-travel to), not just the current one. Compare these to `manifests` and `files` (which only reflect the current snapshot) to see how much old state is accumulating.

In [ ]:
current_manifests = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.manifests").collect()[0][0]
all_manifests = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_manifests").collect()[0][0]

print(f"Manifests in current snapshot: {current_manifests}")
print(f"Manifests across ALL snapshots: {all_manifests}")
print(f"\nThe old {all_manifests - current_manifests} manifests still exist in storage.")

You can see the old manifest files still on disk:

[Open maintenance/taxi_trips/metadata/ in MinIO](http://localhost:9001/browser/warehouse/maintenance/taxi_trips/metadata/)

## Part 3: Data File Compaction

Many small data files hurt scan performance. Each file has overhead for opening, reading metadata, and closing. A full table scan with hundreds of tiny files is slower than scanning a few well-sized files, even though the total data volume is the same. Let's measure this before and after compaction.

### Compact Data Files

### Measure Full Scan Time (Before)

Manifests are already compacted from Part 2, so any difference we see here is purely from data file overhead. Let's benchmark a full aggregation across all 600 small files.

In [ ]:
file_count_before = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.files").collect()[0][0]

scan_times_before = []
for run in range(3):
    start = time.time()
    spark.sql("""
        SELECT COUNT(*), AVG(fare_amount), AVG(trip_distance), AVG(tip_amount)
        FROM polaris.maintenance.taxi_trips
    """).collect()
    scan_times_before.append(time.time() - start)

avg_scan_before = sum(scan_times_before) / len(scan_times_before)
print(f"Full scan with {file_count_before} small files:")
print(f"  3 runs: {', '.join(f'{t:.3f}s' for t in scan_times_before)}")
print(f"  Average: {avg_scan_before:.3f}s")

In [ ]:
print("Compacting data files...")
start = time.time()

spark.sql("""
    CALL polaris.system.rewrite_data_files(
        table => 'maintenance.taxi_trips',
        options => map('target-file-size-bytes', '10485760')
    )
""")

elapsed = time.time() - start
print(f"Data file compaction completed in {elapsed:.2f} seconds")

### Compare Before and After

> **Note on benchmarks:** The timings below illustrate the *relative* impact of compaction on query performance. Absolute numbers will vary based on your local environment. In production with larger datasets and distributed clusters, the improvements from compaction are typically more dramatic.

In [ ]:
print("File statistics AFTER compaction:")
spark.sql("""
    SELECT
        COUNT(*) as file_count,
        ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
        ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
        ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
        ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
    FROM polaris.maintenance.taxi_trips.files
""").show()

### Measure Full Scan Time (After)

In [ ]:
file_count_after = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.files").collect()[0][0]

scan_times_after = []
for run in range(3):
    start = time.time()
    spark.sql("""
        SELECT COUNT(*), AVG(fare_amount), AVG(trip_distance), AVG(tip_amount)
        FROM polaris.maintenance.taxi_trips
    """).collect()
    scan_times_after.append(time.time() - start)

avg_scan_after = sum(scan_times_after) / len(scan_times_after)
print(f"Full scan with {file_count_after} compacted files:")
print(f"  3 runs: {', '.join(f'{t:.3f}s' for t in scan_times_after)}")
print(f"  Average: {avg_scan_after:.3f}s")
print(f"\nScan speedup: {avg_scan_before/avg_scan_after:.1f}x faster")

Far fewer files, much closer to target size. The scan speedup comes from reduced per-file overhead: fewer files to open, read headers from, and close. The total data volume is the same, but the I/O pattern is much more efficient.

In [ ]:
# Note: The micro-batch pattern above uses .limit() which pulls from the head of the
# dataframe and may not cover all days. The partition output below may show fewer
# partitions than the full June dataset. This is expected for demonstrating compaction.
print("Files per partition after compaction:")
spark.sql("""
    SELECT partition, COUNT(*) as file_count,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
    FROM polaris.maintenance.taxi_trips.files
    GROUP BY partition ORDER BY partition
""").show()

Same story with data files. Compaction wrote new large files, but the old small files are still in storage. Until we expire the snapshots that reference them, they won't be cleaned up.

In [ ]:
current_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.files").collect()[0][0]
all_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_data_files").collect()[0][0]

print(f"Data files in current snapshot: {current_files}")
print(f"Data files across ALL snapshots: {all_files}")
print(f"\nCompaction replaced ~600 small files with {current_files} large ones,")
print(f"but the old files ({all_files - current_files:,}) are still on disk, referenced by older snapshots.")

Browse the old small files alongside the new compacted ones:

[Open maintenance/taxi_trips/data/ in MinIO](http://localhost:9001/browser/warehouse/maintenance/taxi_trips/data/)

### Try It: Compact with a Different Target Size

Re-run `rewrite_data_files` with a different `target-file-size-bytes` (e.g., `'5242880'` for 5 MB or `'52428800'` for 50 MB). Compare the resulting file count and average file size. How does the target size affect the number of files per partition?

In [ ]:
# my_target_size = '5242880'  # 5 MB. Try '52428800' (50 MB) too

# spark.sql(f"""
#     CALL polaris.system.rewrite_data_files(
#         table => 'maintenance.taxi_trips',
#         options => map('target-file-size-bytes', '{my_target_size}')
#     )
# """)

# spark.sql("""
#     SELECT COUNT(*) as file_count,
#            ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
#            ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
#            ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb
#     FROM polaris.maintenance.taxi_trips.files
# """).show()

## Part 4: Snapshot Expiration

Every write, compaction, and maintenance operation creates a new **snapshot**. Old snapshots enable time travel, but as we saw above, they also keep old data files and manifests alive in storage, even after compaction replaced them. Expiring old snapshots tells Iceberg it's safe to delete those unreferenced files.

Let's see how much storage our old snapshots are hoarding using the `all_data_files` and `all_manifests` tables we introduced earlier.

### View Snapshot History

In [ ]:
before_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Snapshots BEFORE expiration: {before_count}")

print("\nRecent snapshots:")
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM polaris.maintenance.taxi_trips.snapshots
    ORDER BY committed_at DESC LIMIT 5
""").show(truncate=False)

In [ ]:
current_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.files").collect()[0][0]
all_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_data_files").collect()[0][0]
current_manifests = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.manifests").collect()[0][0]
all_manifests = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_manifests").collect()[0][0]

print(f"{'':20s} {'Current Snapshot':>18s} {'All Snapshots':>15s}")
print("-" * 55)
print(f"{'Data files':<20s} {current_files:>18,} {all_files:>15,}")
print(f"{'Manifests':<20s} {current_manifests:>18,} {all_manifests:>15,}")
print(f"\nOld snapshots are keeping {all_files - current_files:,} extra data files")
print(f"and {all_manifests - current_manifests:,} extra manifests alive in storage.")

The "All Snapshots" column shows every file that **any** snapshot still references. The extra files come from the pre-compaction snapshots. They reference the old small files that were replaced during compaction. Until we expire those snapshots, the old files stay in storage.

[Open maintenance/taxi_trips/data/ in MinIO](http://localhost:9001/browser/warehouse/maintenance/taxi_trips/data/)

### Expire Old Snapshots

In [ ]:
print("Expiring all snapshots except the most recent one...")
start = time.time()

# expire_snapshots keeps the most recent `retain_last` snapshots regardless of age,
# then expires everything else that is older than `older_than`. Setting older_than
# far in the future ensures the age condition doesn't protect any snapshot, so only
# retain_last matters. In production, you'd typically use a reasonable older_than
# (e.g., 7 days) instead. Note: retain_last works on its own too. older_than
# defaults to 5 days when omitted, which is sufficient for most use cases.
spark.sql("""
    CALL polaris.system.expire_snapshots(
        table => 'maintenance.taxi_trips',
        older_than => TIMESTAMP '2099-12-31 23:59:59',
        retain_last => 1
    )
""")

elapsed = time.time() - start
print(f"Snapshot expiration completed in {elapsed:.2f} seconds")

In [ ]:
after_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.maintenance.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Snapshots AFTER expiration: {after_count}")
print(f"Removed {before_count - after_count} snapshots")

In [ ]:
post_all_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_data_files").collect()[0][0]
post_all_manifests = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.all_manifests").collect()[0][0]
post_current_files = spark.sql("SELECT COUNT(*) FROM polaris.maintenance.taxi_trips.files").collect()[0][0]

print(f"{'':20s} {'Before Expire':>15s} {'After Expire':>14s} {'Removed':>10s}")
print("-" * 62)
print(f"{'Snapshots':<20s} {before_count:>15,} {after_count:>14,} {before_count - after_count:>10,}")
print(f"{'All data files':<20s} {all_files:>15,} {post_all_files:>14,} {all_files - post_all_files:>10,}")
print(f"{'All manifests':<20s} {all_manifests:>15,} {post_all_manifests:>14,} {all_manifests - post_all_manifests:>10,}")

files_removed = all_files - post_all_files
manifests_removed = all_manifests - post_all_manifests
print(f"\n{files_removed:,} unreferenced data files physically deleted from storage.")
print(f"{manifests_removed:,} unreferenced manifests physically deleted from storage.")

By keeping only the most recent snapshot (`retain_last => 1`), every old pre-compaction data file and manifest became unreferenced and was physically deleted from storage. The tradeoff: we lose all time travel history. There's only one snapshot left.

In production you'd typically retain more snapshots (e.g., `retain_last => 5` or use `older_than` to keep the last 7 days). The more snapshots you keep, the more old files remain on disk.

Verify in MinIO that the data directory should now contain only the compacted files:

[Open maintenance/taxi_trips/data/ in MinIO](http://localhost:9001/browser/warehouse/maintenance/taxi_trips/data/)

## Part 5: Orphan File Removal

Orphan files occur when writes fail partway through. The data file gets written to storage but is never committed to metadata.

In [ ]:
orphan_key = 'maintenance/taxi_trips/data/orphan-file-12345.parquet'

s3_client.put_object(
    Bucket='warehouse',
    Key=orphan_key,
    Body=b"This is an orphan file that was never committed"
)

print(f"Created fake orphan file: s3://warehouse/{orphan_key}")

### Remove Orphan Files

The SQL procedure `remove_orphan_files` requires the `older_than` timestamp to be at least 24 hours in the past. This safety buffer ensures that files from long-running writes that haven't committed yet aren't accidentally deleted. In practice, most writes commit in seconds, but the margin accounts for very large jobs or retries. In production, the SQL procedure is the standard approach:

```sql
-- Production usage (older_than must be at least 24 hours ago):
-- CALL polaris.system.remove_orphan_files(table => 'maintenance.taxi_trips', older_than => TIMESTAMP '2025-01-01 00:00:00')
```

The procedure deletes any file **older than** the `older_than` timestamp — meaning any file created *before* that time is a candidate. Since we just created our orphan file moments ago, we set `older_than` to one second in the future. Our orphan was created before that future timestamp, so it's eligible for removal. The lab environment's Spark server is configured with `spark.testing=true` to bypass the 24-hour minimum for demonstration purposes — never use this in production.

In [ ]:
from datetime import datetime, timedelta

print("Removing orphan files...")
start = time.time()

# older_than = "delete files created before this time." Setting it to 1 second
# in the future makes our just-created orphan eligible. The lab's Spark server
# has spark.testing=true to bypass the 24-hour minimum (never do this in prod).
cutoff = (datetime.now() + timedelta(seconds=1)).strftime("%Y-%m-%d %H:%M:%S")
spark.sql(f"""
    CALL polaris.system.remove_orphan_files(
        table => 'maintenance.taxi_trips',
        older_than => TIMESTAMP '{cutoff}'
    )
""")

elapsed = time.time() - start
print(f"Orphan file removal completed in {elapsed:.2f} seconds")

In [ ]:
try:
    s3_client.head_object(Bucket='warehouse', Key=orphan_key)
    print("Orphan file still exists. Check the older_than parameter")
except:
    print("Orphan file was successfully removed!")

## Part 6: Monitoring Table Health

After running the full maintenance sequence (compact manifests → compact data files → expire snapshots → remove orphans), how do you verify everything is healthy? Iceberg's metadata tables give you a complete picture:

| Table | Shows | Scope |
|---|---|---|
| `<table>.files` | Data files in the **current** snapshot | What queries read today |
| `<table>.all_data_files` | Data files across **all live** snapshots | Total storage footprint |
| `<table>.manifests` | Manifests for the **current** snapshot | Planning overhead |
| `<table>.all_manifests` | Manifests across **all live** snapshots | Total manifest storage |
| `<table>.snapshots` | All non-expired snapshots | Time travel history |

A well-maintained table should have `files` ≈ `all_data_files` (no old snapshots hoarding replaced files) and a small number of manifests. Let's check our table.

In [ ]:
tbl = "polaris.maintenance.taxi_trips"

file_stats = spark.sql(f"""
    SELECT COUNT(*) as cnt,
           ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_mb,
           ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_mb
    FROM {tbl}.files
""").collect()[0]

all_files = spark.sql(f"SELECT COUNT(*) FROM {tbl}.all_data_files").collect()[0][0]
manifests = spark.sql(f"SELECT COUNT(*) FROM {tbl}.manifests").collect()[0][0]
all_manifests = spark.sql(f"SELECT COUNT(*) FROM {tbl}.all_manifests").collect()[0][0]
snapshots = spark.sql(f"SELECT COUNT(*) FROM {tbl}.snapshots").collect()[0][0]
records = spark.sql(f"SELECT COUNT(*) FROM {tbl}").collect()[0][0]

print("=" * 60)
print("TABLE HEALTH REPORT: polaris.maintenance.taxi_trips")
print("=" * 60)
print(f"\n{'':25s} {'Current':>10s} {'All Snapshots':>15s}")
print("-" * 52)
print(f"{'Data files':<25s} {file_stats['cnt']:>10,} {all_files:>15,}")
print(f"{'Manifests':<25s} {manifests:>10,} {all_manifests:>15,}")
print(f"\nAvg file size: {file_stats['avg_mb']} MB")
print(f"Total data size: {file_stats['total_mb']} MB")
print(f"Snapshots: {snapshots}")
print(f"Records: {records:,}")
print("=" * 60)

if all_files == file_stats['cnt']:
    print("\n✓ Current files = All files. No old snapshots hoarding replaced data.")
else:
    extra = all_files - file_stats['cnt']
    print(f"\n⚠ {extra:,} extra files still referenced by old snapshots.")
    print("  Consider expiring more snapshots to reclaim storage.")

### Try It: Run the Full Maintenance Sequence on Your Own Table

Create a new table, write data in small batches to produce many small files, then apply the full maintenance sequence in the recommended order: (1) compact manifests, (2) compact data files, (3) expire snapshots, (4) remove orphan files. Use the health report query above to monitor improvements at each step.

In [ ]:
# my_table = "polaris.maintenance.my_test_table"

# Step 0: Create a table with a small-file problem
# spark.sql(f"DROP TABLE IF EXISTS {my_table}")
# spark.sql(f"""
#     CREATE TABLE {my_table}
#     USING iceberg PARTITIONED BY (days(tpep_pickup_datetime))
#     AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet` LIMIT 0
# """)
# df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet")
# for i in range(20):
#     df.limit(5000).writeTo(my_table).append()

# Step 1: Compact manifests
# spark.sql(f"CALL polaris.system.rewrite_manifests(table => '{my_table.split('.', 1)[1]}')")

# Step 2: Compact data files
# spark.sql(f"CALL polaris.system.rewrite_data_files(table => '{my_table.split('.', 1)[1]}')")

# Step 3: Expire snapshots
# spark.sql(f"CALL polaris.system.expire_snapshots(table => '{my_table.split('.', 1)[1]}', retain_last => 2)")

# Step 4: Check the health report
# spark.sql(f"SELECT COUNT(*) as files FROM {my_table}.files").show()
# spark.sql(f"SELECT COUNT(*) as snapshots FROM {my_table}.snapshots").show()

# Cleanup
# spark.sql(f"DROP TABLE IF EXISTS {my_table}")

## Cleanup

In [ ]:
# Optional: Drop table
# spark.sql("DROP TABLE IF EXISTS polaris.maintenance.taxi_trips")
# print("Table dropped!")